# 🔧 Feature Engineering — Retail Demand Forecasting

Aggregates POS transactions and inventory snapshots into daily demand features
per store-product combination for demand prediction.

**Reads from:** `silver_pos_transactions`, `silver_products`, `silver_stores`, `silver_inventory_snapshots`

**Writes to:** `gold_ml_features`

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, lit, current_timestamp, when, avg, stddev, count,
    sum as spark_sum, max as spark_max, min as spark_min,
    dayofweek, dayofmonth, month, weekofyear, to_date,
    lag, datediff, round as spark_round
)
from pyspark.sql.window import Window

spark = SparkSession.builder.getOrCreate()
print('Spark session ready')

In [ ]:
# Load Silver tables
txn_df = spark.read.table('silver_pos_transactions')
products_df = spark.read.table('silver_products')
stores_df = spark.read.table('silver_stores')
inv_df = spark.read.table('silver_inventory_snapshots')

print(f'Transactions: {txn_df.count():,} rows')
print(f'Products: {products_df.count():,} rows')
print(f'Stores: {stores_df.count():,} rows')
print(f'Inventory snapshots: {inv_df.count():,} rows')

In [ ]:
# === Daily demand aggregation per store-product ===
daily_demand = (
    txn_df
    .withColumn('txn_date', to_date('transaction_timestamp'))
    .groupBy('store_id', 'product_id', 'txn_date')
    .agg(
        spark_sum('quantity').alias('daily_quantity'),
        spark_sum(col('quantity') * col('unit_price') * (1 - col('discount_pct') / 100)).alias('daily_revenue'),
        count('*').alias('transaction_count'),
        avg('discount_pct').alias('avg_discount'),
        avg('unit_price').alias('avg_price'),
    )
)

print(f'Daily demand rows: {daily_demand.count():,}')

In [ ]:
# === Calendar features ===
daily_features = (
    daily_demand
    .withColumn('day_of_week', dayofweek('txn_date'))
    .withColumn('day_of_month', dayofmonth('txn_date'))
    .withColumn('month', month('txn_date'))
    .withColumn('week_of_year', weekofyear('txn_date'))
    .withColumn('is_weekend', when(dayofweek('txn_date').isin(1, 7), 1).otherwise(0))
)

In [ ]:
# === Lag features (previous day demand as predictor for next day) ===
w = Window.partitionBy('store_id', 'product_id').orderBy('txn_date')

daily_features = (
    daily_features
    .withColumn('demand_lag_1', lag('daily_quantity', 1).over(w))
    .withColumn('demand_lag_7', lag('daily_quantity', 7).over(w))
    .withColumn('revenue_lag_1', lag('daily_revenue', 1).over(w))
    .na.fill(0, ['demand_lag_1', 'demand_lag_7', 'revenue_lag_1'])
)

In [ ]:
# === Join with product and store dimensions ===
ml_features = (
    daily_features
    .join(
        products_df.select('sku', 'category', 'subcategory', 'unit_cost'),
        daily_features['product_id'] == products_df['sku'], 'left'
    )
    .join(
        stores_df.select('store_id', 'region', 'store_format'),
        'store_id', 'left'
    )
    .withColumn('margin_pct',
        spark_round((col('avg_price') - col('unit_cost')) / col('avg_price') * 100, 2)
    )
    .drop('sku')
    .na.fill(0)
    .withColumn('feature_timestamp', current_timestamp())
)

# Target: daily_quantity (what we want to predict)
ml_features.write.mode('overwrite').format('delta').saveAsTable('gold_ml_features')

print(f'\nGold ML features written: {ml_features.count():,} rows')
print(f'Columns: {len(ml_features.columns)}')
ml_features.select('daily_quantity').describe().show()

In [ ]:
spark.sql('OPTIMIZE gold_ml_features')
print('Feature table optimized')